# 01 — TMDB + OMDb → Raw Dataset (movies_clean.csv)

This section collects movie data from TMDB and enriches it from OMDb.
Output: `data/raw/movies_clean.csv`

In [1]:
!pip -q install requests pandas tqdm python-dotenv

import requests
import pandas as pd
import time
from tqdm import tqdm
from pathlib import Path

## Load API Keys from `.env`

Your `.env` file must contain:

TMDB_API_KEY=...
OMDB_API_KEY=...

In [3]:
import sys
from pathlib import Path

# Add parent directory to Python path so we can import from 'src'
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

In [4]:
from src.utils.env import TMDB_API_KEY, OMDB_API_KEY

TMDB_BASE = "https://api.themoviedb.org/3"
OMDB_BASE = "https://www.omdbapi.com/"

In [5]:
def get_json(url, params=None, headers=None, retries=3, sleep=0.7):
    last_err = None
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
            if r.status_code == 200:
                return r.json()
            last_err = f"HTTP {r.status_code}: {r.text[:150]}"
        except Exception as e:
            last_err = str(e)
        time.sleep(sleep)
    return {"_error": last_err}

In [6]:
def tmdb_discover_movies(page=1, sort_by="popularity.desc", min_votes=200):
    url = f"{TMDB_BASE}/discover/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "page": page,
        "sort_by": sort_by,
        "vote_count.gte": min_votes,
        "include_adult": "false",
        "include_video": "false",
        "language": "en-US"
    }
    return get_json(url, params=params)

In [7]:
def tmdb_movie_details(movie_id):
    url = f"{TMDB_BASE}/movie/{movie_id}"
    params = {"api_key": TMDB_API_KEY, "language": "en-US"}
    return get_json(url, params=params)

def tmdb_movie_credits(movie_id):
    url = f"{TMDB_BASE}/movie/{movie_id}/credits"
    params = {"api_key": TMDB_API_KEY, "language": "en-US"}
    return get_json(url, params=params)

def extract_director_and_cast(credits_json, top_n_cast=5):
    director = None
    cast_names = []
    if isinstance(credits_json, dict):
        crew = credits_json.get("crew", []) or []
        for person in crew:
            if person.get("job") == "Director":
                director = person.get("name")
                break
        cast = credits_json.get("cast", []) or []
        cast_names = [c.get("name") for c in cast[:top_n_cast] if c.get("name")]
    return director, ", ".join(cast_names)

In [8]:
def omdb_by_imdb_id(imdb_id):
    params = {"apikey": OMDB_API_KEY, "i": imdb_id, "plot": "short"}
    return get_json(OMDB_BASE, params=params)

def safe_float(x):
    try:
        return float(x)
    except:
        return None

def safe_int_from_votes(v):
    try:
        return int(str(v).replace(",", ""))
    except:
        return None

In [9]:
TARGET_MOVIES = 800
PAGES_TO_TRY = 200

rows = []
seen_tmdb_ids = set()

for page in tqdm(range(1, PAGES_TO_TRY + 1), desc="TMDB pages"):
    data = tmdb_discover_movies(page=page, sort_by="popularity.desc", min_votes=200)
    results = data.get("results", []) if isinstance(data, dict) else []
    if not results:
        break

    for m in results:
        mid = m.get("id")
        if not mid or mid in seen_tmdb_ids:
            continue
        seen_tmdb_ids.add(mid)

        rows.append({
            "tmdb_id": mid,
            "title": m.get("title"),
            "release_date": m.get("release_date"),
            "overview": m.get("overview"),
            "tmdb_vote_average": m.get("vote_average"),
            "tmdb_vote_count": m.get("vote_count"),
            "popularity": m.get("popularity"),
            "original_language": m.get("original_language"),
            "poster_path": m.get("poster_path")
        })

        if len(rows) >= TARGET_MOVIES:
            break

    if len(rows) >= TARGET_MOVIES:
        break

tmdb_list_df = pd.DataFrame(rows)
tmdb_list_df.head()

TMDB pages:  20%|█▉        | 39/200 [00:24<01:42,  1.57it/s]


,tmdb_id,title,release_date,overview,tmdb_vote_average,tmdb_vote_count,popularity,original_language,poster_path
0,1290821,Shelter,2026-01-28,A man living in self-imposed exile on a remote...,6.781,212,388.3942,en,/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg
1,10156,History of the World: Part I,1981-06-12,An uproarious version of history that proves n...,6.762,887,275.8545,en,/6iAl78qZHT65erPXr2YW6Y54wlY.jpg
2,1159559,Scream 7,2026-02-25,When a new Ghostface killer emerges in the qui...,6.000,230,179.2451,en,/jjyuk0edLiW8vOSnlfwWCCLpbh5.jpg
3,680493,Return to Silent Hill,2026-01-21,When James receives a mysterious letter from h...,5.051,237,162.2613,en,/fqAGFN2K2kDL0EHxvJaXzaMUkkt.jpg
4,1266798,Dog 51,2025-10-15,"In the near future, Paris is divided into thre...",6.252,214,117.5584,fr,/olMvhbCRuhTbf6EMKxKGHi2on3L.jpg


In [ ]:
enriched = []

for _, row in tqdm(tmdb_list_df.iterrows(), total=len(tmdb_list_df), desc="Enrich movies"):
    mid = int(row["tmdb_id"])
    details = tmdb_movie_details(mid)
    credits = tmdb_movie_credits(mid)

    imdb_id = details.get("imdb_id") if isinstance(details, dict) else None
    runtime = details.get("runtime") if isinstance(details, dict) else None

    genres = None
    if isinstance(details, dict):
        g = details.get("genres", []) or []
        genres = ", ".join([x.get("name") for x in g if x.get("name")])

    director, top_cast = extract_director_and_cast(credits, top_n_cast=5)

    imdb_rating = imdb_votes = metascore = None
    rated = awards = box_office = None
    omdb_genre = omdb_plot = None

    if imdb_id and isinstance(imdb_id, str) and imdb_id.startswith("tt"):
        om = omdb_by_imdb_id(imdb_id)
        if isinstance(om, dict) and om.get("Response") == "True":
            imdb_rating = safe_float(om.get("imdbRating"))
            imdb_votes = safe_int_from_votes(om.get("imdbVotes"))
            metascore = safe_int_from_votes(om.get("Metascore"))
            rated = om.get("Rated")
            awards = om.get("Awards")
            box_office = om.get("BoxOffice")
            omdb_genre = om.get("Genre")
            omdb_plot = om.get("Plot")

    poster_url = None
    if row.get("poster_path"):
        poster_url = f"https://image.tmdb.org/t/p/w500{row['poster_path']}"

    enriched.append({
        "tmdb_id": mid,
        "imdb_id": imdb_id,
        "title": row.get("title"),
        "release_date": row.get("release_date"),
        "year": (str(row.get("release_date"))[:4] if row.get("release_date") else None),
        "runtime": runtime,
        "genres_tmdb": genres,
        "genres_omdb": omdb_genre,
        "overview_tmdb": row.get("overview"),
        "plot_omdb": omdb_plot,
        "director": director,
        "top_cast": top_cast,
        "tmdb_rating": row.get("tmdb_vote_average"),
        "tmdb_vote_count": row.get("tmdb_vote_count"),
        "popularity": row.get("popularity"),
        "imdb_rating": imdb_rating,
        "imdb_votes": imdb_votes,
        "metascore": metascore,
        "rated": rated,
        "awards": awards,
        "box_office": box_office,
        "poster_url": poster_url
    })

movies_df = pd.DataFrame(enriched)

# Use project root path
raw_path = Path().resolve().parent / "data" / "raw" / "movies_clean.csv"
raw_path.parent.mkdir(parents=True, exist_ok=True)

movies_df["year"] = pd.to_numeric(movies_df["year"], errors="coerce")
movies_df = movies_df.drop_duplicates(subset=["tmdb_id"]).reset_index(drop=True)
movies_df = movies_df[movies_df["title"].notna()].reset_index(drop=True)

movies_df.to_csv(raw_path, index=False)
print("Saved raw dataset:", raw_path, "| rows:", len(movies_df))
movies_df.head()

Enrich movies: 100%|██████████| 800/800 [34:33<00:00,  2.59s/it]  


Saved raw dataset: data/raw/movies_clean.csv | rows: 800


,tmdb_id,imdb_id,title,release_date,year,runtime,genres_tmdb,genres_omdb,overview_tmdb,plot_omdb,...,tmdb_rating,tmdb_vote_count,popularity,imdb_rating,imdb_votes,metascore,rated,awards,box_office,poster_url
0,1290821,tt32357218,Shelter,2026-01-28,2026,107,"Action, Crime, Thriller","Action, Thriller",A man living in self-imposed exile on a remote...,Michael Mason is a recluse on a remote Scottis...,...,6.781,212,388.3942,6.4,5906.0,50.0,R,N/A,"$12,377,905",https://image.tmdb.org/t/p/w500/buPFnHZ3xQy6vZ...
1,10156,tt0082517,History of the World: Part I,1981-06-12,1981,92,Comedy,"Comedy, History, Musical",An uproarious version of history that proves n...,Mel Brooks brings his one-of-a-kind comic touc...,...,6.762,887,275.8545,6.8,58818.0,47.0,R,1 win & 1 nomination total,"$31,672,907",https://image.tmdb.org/t/p/w500/6iAl78qZHT65er...
2,1159559,tt27047903,Scream 7,2026-02-25,2026,114,"Horror, Mystery, Crime","Horror, Mystery",When a new Ghostface killer emerges in the qui...,When a new Ghostface killer emerges in the tow...,...,6.000,230,179.2451,NaN,NaN,NaN,R,N/A,N/A,https://image.tmdb.org/t/p/w500/jjyuk0edLiW8vO...
3,680493,tt22868010,Return to Silent Hill,2026-01-21,2026,106,"Mystery, Drama, Horror","Drama, Horror, Mystery",When James receives a mysterious letter from h...,When a man receives a mysterious letter from h...,...,5.051,237,162.2613,4.1,10544.0,34.0,R,N/A,"$5,544,971",https://image.tmdb.org/t/p/w500/fqAGFN2K2kDL0E...
4,1266798,tt33255286,Dog 51,2025-10-15,2025,100,"Thriller, Science Fiction, Crime, Action","Crime, Sci-Fi, Thriller","In the near future, Paris is divided into thre...","In a future Paris divided by class, an AI name...",...,6.252,214,117.5584,5.8,1751.0,NaN,N/A,3 nominations total,N/A,https://image.tmdb.org/t/p/w500/olMvhbCRuhTbf6...


# 02 — Generic ETL on Any Movie CSV (movies_transformed.csv)

This section transforms **any** movie CSV into a standard format.
Input: `data/raw/movies_clean.csv` (or any other movie csv path)
Output: `data/processed/movies_transformed.csv`

In [ ]:
from src.etl.extract import extract_csv
from src.etl.transform import transform_movies
from src.etl.load import load_csv

# Use project root paths
input_path = Path().resolve().parent / "data" / "raw" / "movies_clean.csv"
output_path = Path().resolve().parent / "data" / "processed" / "movies_transformed.csv"

df_raw = extract_csv(str(input_path))
df_clean = transform_movies(df_raw)
load_csv(df_clean, str(output_path))

print("Saved processed dataset:", output_path, "| rows:", len(df_clean))
df_clean.head()

Saved processed dataset: data/processed/movies_transformed.csv | rows: 800


,title,year,release_date,genres,overview,runtime,imdb_id,tmdb_id,imdb_rating,imdb_votes,tmdb_rating,tmdb_vote_count,popularity,director,cast,poster_url
0,Shelter,2026,2026-01-28,"Action, Crime, Thriller",Michael Mason is a recluse on a remote Scottis...,107,tt32357218,1290821,6.4,5906.0,6.781,212,388.3942,Ric Roman Waugh,"Jason Statham, Bodhi Rae Breathnach, Michael S...",https://image.tmdb.org/t/p/w500/buPFnHZ3xQy6vZ...
1,History of the World: Part I,1981,1981-06-12,Comedy,Mel Brooks brings his one-of-a-kind comic touc...,92,tt0082517,10156,6.8,58818.0,6.762,887,275.8545,Mel Brooks,"Mel Brooks, Dom DeLuise, Madeline Kahn, Harvey...",https://image.tmdb.org/t/p/w500/6iAl78qZHT65er...
2,Scream 7,2026,2026-02-25,"Crime, Horror, Mystery",When a new Ghostface killer emerges in the tow...,114,tt27047903,1159559,NaN,NaN,6.000,230,179.2451,Kevin Williamson,"Neve Campbell, Courteney Cox, Isabel May, Jasm...",https://image.tmdb.org/t/p/w500/jjyuk0edLiW8vO...
3,Return to Silent Hill,2026,2026-01-21,"Drama, Horror, Mystery",When a man receives a mysterious letter from h...,106,tt22868010,680493,4.1,10544.0,5.051,237,162.2613,Christophe Gans,"Jeremy Irvine, Hannah Emily Anderson, Evie Tem...",https://image.tmdb.org/t/p/w500/fqAGFN2K2kDL0E...
4,Dog 51,2025,2025-10-15,"Action, Crime, Science Fiction, Thriller","In a future Paris divided by class, an AI name...",100,tt33255286,1266798,5.8,1751.0,6.252,214,117.5584,Cédric Jimenez,"Gilles Lellouche, Adèle Exarchopoulos, Louis G...",https://image.tmdb.org/t/p/w500/olMvhbCRuhTbf6...


# 03 — Recommender Test (Notebook)
We will test recommendations using the processed dataset.

In [ ]:
from src.recommender.build_features import build_feature_text
from src.recommender.recommend import MovieRecommender

# Use project root path
processed_path = Path().resolve().parent / "data" / "processed" / "movies_transformed.csv"

df = pd.read_csv(processed_path)
df["feature_text"] = build_feature_text(df)

model = MovieRecommender(df, feature_col="feature_text")
model.recommend("Inception", top_n=10)

,title,similarity,year,genres,imdb_rating,tmdb_rating,popularity,poster_url
0,The Dark Knight Rises,0.151399,2012,"Action, Crime, Drama, Thriller",8.4,7.792,19.7980,https://image.tmdb.org/t/p/w500/hr0L2aueqlP2BY...
1,The Revenant,0.103364,2015,"Adventure, Drama, Western",8.0,7.539,22.9298,https://image.tmdb.org/t/p/w500/ji3ecJphATlVgW...
2,Godzilla,0.096939,2014,"Action, Drama, Science Fiction",6.4,6.353,19.2387,https://image.tmdb.org/t/p/w500/tphkjmQq8WebuV...
3,Interstellar,0.095390,2014,"Adventure, Drama, Science Fiction",8.7,8.468,50.3921,https://image.tmdb.org/t/p/w500/gEU2QniE6E77NI...
4,Looper,0.093332,2012,"Action, Science Fiction, Thriller",7.4,6.902,19.4795,https://image.tmdb.org/t/p/w500/sNjL6SqErDBE8O...
5,Fantastic Four,0.080240,2005,"Action, Fantasy, Science Fiction",5.7,5.804,15.2782,https://image.tmdb.org/t/p/w500/8HLQLILZLhDQWO...
6,The Creator,0.075874,2023,"Action, Adventure, Science Fiction",6.7,7.012,20.5662,https://image.tmdb.org/t/p/w500/3dSivDtOuyxLDx...
7,The Dark Knight,0.072428,2008,"Action, Crime, Thriller",9.1,8.527,40.8598,https://image.tmdb.org/t/p/w500/qJ2tW6WMUDux91...
8,Shutter Island,0.069703,2010,"Drama, Mystery, Thriller",8.2,8.200,21.9957,https://image.tmdb.org/t/p/w500/nrmXQ0zcZUL8jF...
9,Mysterious Skin,0.066301,2005,Drama,7.6,7.700,17.5675,https://image.tmdb.org/t/p/w500/wgSwDqhl6Rt3cu...
